# cuda-matmul: SGEMM from naive to vectorized

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.


In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
!mkdir -p cuda-matmul/src

In [ ]:
%%writefile cuda-matmul/Makefile
# ARCH=native targets the GPU in this machine (CUDA 11.6+). On older toolkits use e.g. ARCH=sm_75 (T4).
ARCH ?= native
NVCC ?= nvcc
FLAGS = -O3 -std=c++17 -arch=$(ARCH) -lineinfo

sgemm: src/main.cu src/kernels.cuh
	$(NVCC) $(FLAGS) -o $@ src/main.cu -lcublas

run: sgemm
	./sgemm

clean:
	rm -f sgemm

.PHONY: run clean


In [ ]:
%%writefile cuda-matmul/src/kernels.cuh
// SGEMM kernels, from naive to register-blocked and vectorized.
// All kernels compute C = alpha * A * B + beta * C for row-major A (MxK), B (KxN), C (MxN).
// Kernels 4-6 assume M, N, K are multiples of their block tile sizes (main.cu checks this).
#pragma once

// 1. Naive: one thread per output element.
//    threadIdx.x walks down rows, so neighbouring threads in a warp touch addresses K floats apart.
__global__ void sgemm_naive(int M, int N, int K, float alpha, const float *A, const float *B,
                            float beta, float *C) {
  const int row = blockIdx.x * blockDim.x + threadIdx.x;
  const int col = blockIdx.y * blockDim.y + threadIdx.y;
  if (row < M && col < N) {
    float acc = 0.f;
    for (int k = 0; k < K; ++k) acc += A[row * K + k] * B[k * N + col];
    C[row * N + col] = alpha * acc + beta * C[row * N + col];
  }
}

// 2. Coalesced: same math, but threadIdx.x now walks along columns, so a warp reads
//    32 consecutive floats of B and writes 32 consecutive floats of C.
__global__ void sgemm_coalesced(int M, int N, int K, float alpha, const float *A, const float *B,
                                float beta, float *C) {
  const int col = blockIdx.x * blockDim.x + threadIdx.x;
  const int row = blockIdx.y * blockDim.y + threadIdx.y;
  if (row < M && col < N) {
    float acc = 0.f;
    for (int k = 0; k < K; ++k) acc += A[row * K + k] * B[k * N + col];
    C[row * N + col] = alpha * acc + beta * C[row * N + col];
  }
}

// 3. Shared-memory tiling: each block stages a TxT tile of A and of B in shared memory,
//    so every global load is reused T times instead of once.
template <int T>
__global__ void sgemm_smem(int M, int N, int K, float alpha, const float *A, const float *B,
                           float beta, float *C) {
  __shared__ float As[T][T];
  __shared__ float Bs[T][T];
  const int tx = threadIdx.x, ty = threadIdx.y;
  const int row = blockIdx.y * T + ty;
  const int col = blockIdx.x * T + tx;
  float acc = 0.f;
  for (int k0 = 0; k0 < K; k0 += T) {
    As[ty][tx] = (row < M && k0 + tx < K) ? A[row * K + k0 + tx] : 0.f;
    Bs[ty][tx] = (k0 + ty < K && col < N) ? B[(k0 + ty) * N + col] : 0.f;
    __syncthreads();
    for (int k = 0; k < T; ++k) acc += As[ty][k] * Bs[k][tx];
    __syncthreads();
  }
  if (row < M && col < N) C[row * N + col] = alpha * acc + beta * C[row * N + col];
}

// 4. 1D block tiling: each thread computes TM outputs down a column, so one value loaded
//    from Bs is reused TM times out of a register.
//    Launch with BM*BN/TM threads; needs BM*BK == BK*BN == thread count.
template <int BM, int BN, int BK, int TM>
__global__ void sgemm_1d_blocktile(int M, int N, int K, float alpha, const float *A,
                                   const float *B, float beta, float *C) {
  const int cRow = blockIdx.y, cCol = blockIdx.x;
  const int threadCol = threadIdx.x % BN;
  const int threadRow = threadIdx.x / BN;
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];
  A += cRow * BM * K;
  B += cCol * BN;
  C += cRow * BM * N + cCol * BN;
  const int innerColA = threadIdx.x % BK, innerRowA = threadIdx.x / BK;
  const int innerColB = threadIdx.x % BN, innerRowB = threadIdx.x / BN;
  float res[TM] = {0.f};
  for (int bk = 0; bk < K; bk += BK) {
    As[innerRowA * BK + innerColA] = A[innerRowA * K + innerColA];
    Bs[innerRowB * BN + innerColB] = B[innerRowB * N + innerColB];
    __syncthreads();
    A += BK;
    B += BK * N;
    for (int d = 0; d < BK; ++d) {
      const float b = Bs[d * BN + threadCol];
      for (int i = 0; i < TM; ++i) res[i] += As[(threadRow * TM + i) * BK + d] * b;
    }
    __syncthreads();
  }
  for (int i = 0; i < TM; ++i) {
    const int idx = (threadRow * TM + i) * N + threadCol;
    C[idx] = alpha * res[i] + beta * C[idx];
  }
}

// 5. 2D block tiling: each thread computes a TMxTN tile using an outer product of
//    TM values of A and TN values of B held in registers (TM*TN FMAs per TM+TN loads).
//    This is the GPU version of the 8x8 register-blocked micro-kernel in fast-matmul.
template <int BM, int BN, int BK, int TM, int TN>
__global__ void __launch_bounds__((BM * BN) / (TM * TN))
    sgemm_2d_blocktile(int M, int N, int K, float alpha, const float *A, const float *B,
                       float beta, float *C) {
  constexpr int nThreads = (BM * BN) / (TM * TN);
  const int cRow = blockIdx.y, cCol = blockIdx.x;
  const int threadCol = threadIdx.x % (BN / TN);
  const int threadRow = threadIdx.x / (BN / TN);
  __shared__ float As[BM * BK];
  __shared__ float Bs[BK * BN];
  A += cRow * BM * K;
  B += cCol * BN;
  C += cRow * BM * N + cCol * BN;
  const int innerRowA = threadIdx.x / BK, innerColA = threadIdx.x % BK;
  constexpr int strideA = nThreads / BK;
  const int innerRowB = threadIdx.x / BN, innerColB = threadIdx.x % BN;
  constexpr int strideB = nThreads / BN;
  float res[TM * TN] = {0.f};
  float regM[TM], regN[TN];
  for (int bk = 0; bk < K; bk += BK) {
    for (int o = 0; o < BM; o += strideA)
      As[(innerRowA + o) * BK + innerColA] = A[(innerRowA + o) * K + innerColA];
    for (int o = 0; o < BK; o += strideB)
      Bs[(innerRowB + o) * BN + innerColB] = B[(innerRowB + o) * N + innerColB];
    __syncthreads();
    A += BK;
    B += BK * N;
    for (int d = 0; d < BK; ++d) {
      for (int i = 0; i < TM; ++i) regM[i] = As[(threadRow * TM + i) * BK + d];
      for (int j = 0; j < TN; ++j) regN[j] = Bs[d * BN + threadCol * TN + j];
      for (int i = 0; i < TM; ++i)
        for (int j = 0; j < TN; ++j) res[i * TN + j] += regM[i] * regN[j];
    }
    __syncthreads();
  }
  for (int i = 0; i < TM; ++i)
    for (int j = 0; j < TN; ++j) {
      const int idx = (threadRow * TM + i) * N + threadCol * TN + j;
      C[idx] = alpha * res[i * TN + j] + beta * C[idx];
    }
}

// 6. Vectorized: kernel 5 plus 128-bit (float4) global loads and stores, and A stored
//    transposed in shared memory so the inner loop reads As with unit stride.
//    Needs BM*BK/4 == BK*BN/4 == thread count (128x128x8 tile, 256 threads).
template <int BM, int BN, int BK, int TM, int TN>
__global__ void __launch_bounds__((BM * BN) / (TM * TN))
    sgemm_vectorized(int M, int N, int K, float alpha, const float *A, const float *B,
                     float beta, float *C) {
  const int cRow = blockIdx.y, cCol = blockIdx.x;
  const int threadCol = threadIdx.x % (BN / TN);
  const int threadRow = threadIdx.x / (BN / TN);
  __shared__ float As[BK * BM];  // transposed: As[k * BM + m]
  __shared__ float Bs[BK * BN];
  A += cRow * BM * K;
  B += cCol * BN;
  C += cRow * BM * N + cCol * BN;
  const int innerRowA = threadIdx.x / (BK / 4), innerColA = threadIdx.x % (BK / 4);
  const int innerRowB = threadIdx.x / (BN / 4), innerColB = threadIdx.x % (BN / 4);
  float res[TM * TN] = {0.f};
  float regM[TM], regN[TN];
  for (int bk = 0; bk < K; bk += BK) {
    const float4 a = reinterpret_cast<const float4 *>(&A[innerRowA * K + innerColA * 4])[0];
    As[(innerColA * 4 + 0) * BM + innerRowA] = a.x;
    As[(innerColA * 4 + 1) * BM + innerRowA] = a.y;
    As[(innerColA * 4 + 2) * BM + innerRowA] = a.z;
    As[(innerColA * 4 + 3) * BM + innerRowA] = a.w;
    reinterpret_cast<float4 *>(&Bs[innerRowB * BN + innerColB * 4])[0] =
        reinterpret_cast<const float4 *>(&B[innerRowB * N + innerColB * 4])[0];
    __syncthreads();
    A += BK;
    B += BK * N;
    for (int d = 0; d < BK; ++d) {
      for (int i = 0; i < TM; ++i) regM[i] = As[d * BM + threadRow * TM + i];
      for (int j = 0; j < TN; ++j) regN[j] = Bs[d * BN + threadCol * TN + j];
      for (int i = 0; i < TM; ++i)
        for (int j = 0; j < TN; ++j) res[i * TN + j] += regM[i] * regN[j];
    }
    __syncthreads();
  }
  for (int i = 0; i < TM; ++i)
    for (int j = 0; j < TN; j += 4) {
      float4 *dst = reinterpret_cast<float4 *>(&C[(threadRow * TM + i) * N + threadCol * TN + j]);
      float4 c = dst[0];
      c.x = alpha * res[i * TN + j + 0] + beta * c.x;
      c.y = alpha * res[i * TN + j + 1] + beta * c.y;
      c.z = alpha * res[i * TN + j + 2] + beta * c.z;
      c.w = alpha * res[i * TN + j + 3] + beta * c.w;
      dst[0] = c;
    }
}


In [ ]:
%%writefile cuda-matmul/src/main.cu
// Benchmark harness: checks every kernel against cuBLAS, then times it.
// Usage: ./sgemm [size ...]   (default sizes: 1024 2048 4096; each must be a multiple of 128)
#include <cublas_v2.h>
#include <cuda_runtime.h>

#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <random>
#include <vector>

#include "kernels.cuh"

#define CUDA_CHECK(x)                                                                     \
  do {                                                                                    \
    cudaError_t err = (x);                                                                \
    if (err != cudaSuccess) {                                                             \
      fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err), __FILE__, __LINE__); \
      exit(1);                                                                            \
    }                                                                                     \
  } while (0)

#define CUBLAS_CHECK(x)                                                     \
  do {                                                                      \
    cublasStatus_t st = (x);                                                \
    if (st != CUBLAS_STATUS_SUCCESS) {                                      \
      fprintf(stderr, "cuBLAS error %d at %s:%d\n", st, __FILE__, __LINE__); \
      exit(1);                                                              \
    }                                                                       \
  } while (0)

constexpr int kWarmup = 2;
constexpr int kReps = 10;

static cublasHandle_t g_handle;

inline int cdiv(int a, int b) { return (a + b - 1) / b; }

// Launches kernel `id` once. Kernel 0 is cuBLAS.
void run(int id, int M, int N, int K, float alpha, const float *A, const float *B, float beta,
         float *C) {
  switch (id) {
    case 0:
      // cuBLAS is column-major. A row-major MxN matrix is a column-major NxM matrix, so
      // computing C^T = B^T A^T in column-major gives C = A B in row-major.
      CUBLAS_CHECK(cublasSgemm(g_handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, B, N, A, K,
                               &beta, C, N));
      break;
    case 1:
      sgemm_naive<<<dim3(cdiv(M, 32), cdiv(N, 32)), dim3(32, 32)>>>(M, N, K, alpha, A, B, beta, C);
      break;
    case 2:
      sgemm_coalesced<<<dim3(cdiv(N, 32), cdiv(M, 32)), dim3(32, 32)>>>(M, N, K, alpha, A, B, beta, C);
      break;
    case 3:
      sgemm_smem<32><<<dim3(cdiv(N, 32), cdiv(M, 32)), dim3(32, 32)>>>(M, N, K, alpha, A, B, beta, C);
      break;
    case 4: {
      constexpr int BM = 64, BN = 64, BK = 8, TM = 8;
      sgemm_1d_blocktile<BM, BN, BK, TM>
          <<<dim3(N / BN, M / BM), (BM * BN) / TM>>>(M, N, K, alpha, A, B, beta, C);
      break;
    }
    case 5: {
      constexpr int BM = 128, BN = 128, BK = 8, TM = 8, TN = 8;
      sgemm_2d_blocktile<BM, BN, BK, TM, TN>
          <<<dim3(N / BN, M / BM), (BM * BN) / (TM * TN)>>>(M, N, K, alpha, A, B, beta, C);
      break;
    }
    case 6: {
      constexpr int BM = 128, BN = 128, BK = 8, TM = 8, TN = 8;
      sgemm_vectorized<BM, BN, BK, TM, TN>
          <<<dim3(N / BN, M / BM), (BM * BN) / (TM * TN)>>>(M, N, K, alpha, A, B, beta, C);
      break;
    }
  }
}

const char *kNames[] = {"cuBLAS", "1 naive", "2 coalesced", "3 shared-mem tiling",
                        "4 1D block tiling", "5 2D block tiling", "6 vectorized float4"};
constexpr int kNumKernels = 7;

int main(int argc, char **argv) {
  std::vector<int> sizes;
  for (int i = 1; i < argc; ++i) sizes.push_back(atoi(argv[i]));
  if (sizes.empty()) sizes = {1024, 2048, 4096};

  cudaDeviceProp prop;
  CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
  printf("GPU: %s (sm_%d%d, %d SMs)\n\n", prop.name, prop.major, prop.minor,
         prop.multiProcessorCount);
  CUBLAS_CHECK(cublasCreate(&g_handle));

  const float alpha = 1.f, beta = 0.f;
  std::mt19937 rng(42);
  std::uniform_real_distribution<float> dist(-1.f, 1.f);

  for (int n : sizes) {
    if (n % 128 != 0) {
      fprintf(stderr, "skipping %d: sizes must be multiples of 128\n", n);
      continue;
    }
    const int M = n, N = n, K = n;
    const size_t bytesA = sizeof(float) * M * K, bytesB = sizeof(float) * K * N,
                 bytesC = sizeof(float) * M * N;
    std::vector<float> hA(M * K), hB(K * N), hRef(M * N), hC(M * N);
    for (auto &x : hA) x = dist(rng);
    for (auto &x : hB) x = dist(rng);

    float *dA, *dB, *dC, *dRef;
    CUDA_CHECK(cudaMalloc(&dA, bytesA));
    CUDA_CHECK(cudaMalloc(&dB, bytesB));
    CUDA_CHECK(cudaMalloc(&dC, bytesC));
    CUDA_CHECK(cudaMalloc(&dRef, bytesC));
    CUDA_CHECK(cudaMemcpy(dA, hA.data(), bytesA, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB.data(), bytesB, cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemset(dRef, 0, bytesC));
    run(0, M, N, K, alpha, dA, dB, beta, dRef);
    CUDA_CHECK(cudaMemcpy(hRef.data(), dRef, bytesC, cudaMemcpyDeviceToHost));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    printf("M = N = K = %d\n", n);
    printf("  %-22s %10s %10s %12s  %s\n", "kernel", "ms", "GFLOP/s", "% of cuBLAS", "check");
    double cublasGflops = 0;
    for (int id = 0; id < kNumKernels; ++id) {
      // Correctness: one run from a zeroed C, compared to cuBLAS.
      CUDA_CHECK(cudaMemset(dC, 0, bytesC));
      run(id, M, N, K, alpha, dA, dB, beta, dC);
      CUDA_CHECK(cudaGetLastError());
      CUDA_CHECK(cudaDeviceSynchronize());
      CUDA_CHECK(cudaMemcpy(hC.data(), dC, bytesC, cudaMemcpyDeviceToHost));
      double maxErr = 0;
      bool ok = true;
      for (int i = 0; i < M * N; ++i) {
        const double diff = std::fabs(hC[i] - hRef[i]);
        maxErr = std::fmax(maxErr, diff);
        if (!(diff <= 1e-2 + 1e-3 * std::fabs(hRef[i]))) ok = false;  // also catches NaN
      }

      // Timing: warm up, then average kReps back-to-back launches.
      for (int r = 0; r < kWarmup; ++r) run(id, M, N, K, alpha, dA, dB, beta, dC);
      CUDA_CHECK(cudaEventRecord(start));
      for (int r = 0; r < kReps; ++r) run(id, M, N, K, alpha, dA, dB, beta, dC);
      CUDA_CHECK(cudaEventRecord(stop));
      CUDA_CHECK(cudaEventSynchronize(stop));
      float ms = 0;
      CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
      ms /= kReps;
      const double gflops = 2.0 * M * N * K / (ms * 1e6);
      if (id == 0) cublasGflops = gflops;
      printf("  %-22s %10.3f %10.1f %11.1f%%  %s (max err %.2e)\n", kNames[id], ms, gflops,
             100.0 * gflops / cublasGflops, ok ? "PASS" : "FAIL", maxErr);
    }
    printf("\n");

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    CUDA_CHECK(cudaFree(dRef));
  }
  CUBLAS_CHECK(cublasDestroy(g_handle));
  return 0;
}


In [ ]:
!cd cuda-matmul && make ARCH=native

In [ ]:
!cd cuda-matmul && ./sgemm 1024 2048 4096

Copy the table above into `RESULTS.md` (GPU name, sizes, GFLOP/s, % of cuBLAS).

Optional profiling: `!cd cuda-matmul && ncu --set basic -k regex:sgemm_2d ./sgemm 2048`